# 店舗売上需要予測: 実験 #013 店舗・商品 × 曜日別ベースライン ＆ 再帰推論

- **コンペ名:** [Store Sales - Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
- **目的:** 
  1. スーパーの売上において最も頑健な周期性である「曜日×店舗」「曜日×商品」のベースライン水準を注入
  2. `dow_mean_4w`: 当該系列の過去4週間（Lag 7, 14, 21, 28）対数売上平均
  3. `dow_decay_mean_4w`: 直近の週を重視した同曜日の減衰加重平均
  4. `dow_diff_lag1`: 前日売上と同曜日平均との差分ギャップ
  5. `store_dow_mean_4w`: 直前4週間における店舗全体の曜日別平均売上（混雑プロファイル）
  6. `family_dow_mean_4w`: 直前4週間におけるカテゴリ全体の曜日別平均売上（需要プロファイル）
  7. テスト期間（16日間）は直前28日間の確定実績から固定ベースラインを安全にマージ（未来リーク防止）
- **モデル:** 33商品ファミリー別 3大GBDTアンサンブル (LightGBM 50% + CatBoost 30% + XGBoost 20%)
- **ハードウェア最適化:** Intel N100 (4コア/4スレッド CPU 最適化: `n_jobs=4`, `thread_count=4`)
- **評価指標:** **RMSLE (Root Mean Squared Logarithmic Error)**
- **結果:** 手元検証 **Val RMSLE `0.39962`**、Public LB **`0.38972`**（単体モデルとして本番最高峰 0.38 台を安定記録！）

## 1. ライブラリ読み込み & データ準備

In [ ]:
import os
import warnings
from datetime import timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

warnings.filterwarnings('ignore', category=UserWarning)

base_dir = '../'
train_path = os.path.join(base_dir, 'data', 'train.csv')
test_path = os.path.join(base_dir, 'data', 'test.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

## 2. 曜日グループ平均（Day-of-Week Group Mean）特徴量の構築

In [ ]:
test_df['sales'] = np.nan
all_df = pd.concat([train_df, test_df], ignore_index=True)
all_df['log_sales'] = np.log1p(all_df['sales'])
all_df['dayofweek'] = all_df['date'].dt.dayofweek

# 店舗別・ファミリー別の曜日平均 (4週間 rolling: shift(1) で未来リーク完全防止)
df_store_dow = all_df[all_df['date'] >= '2016-01-01'].groupby(['date', 'store_nbr', 'dayofweek'])['log_sales'].mean().reset_index()
df_store_dow = df_store_dow.sort_values(['store_nbr', 'dayofweek', 'date']).reset_index(drop=True)
df_store_dow['store_dow_mean_4w'] = (
    df_store_dow.groupby(['store_nbr', 'dayofweek'])['log_sales']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
)

df_fam_dow = all_df[all_df['date'] >= '2016-01-01'].groupby(['date', 'family', 'dayofweek'])['log_sales'].mean().reset_index()
df_fam_dow = df_fam_dow.sort_values(['family', 'dayofweek', 'date']).reset_index(drop=True)
df_fam_dow['family_dow_mean_4w'] = (
    df_fam_dow.groupby(['family', 'dayofweek'])['log_sales']
    .transform(lambda x: x.shift(1).rolling(4, min_periods=1).mean())
)

# テスト期間用ベースライン補間
train_mask_for_base = (all_df['date'] >= '2017-07-19') & (all_df['date'] <= '2017-08-15')
base_store_dow = all_df[train_mask_for_base].groupby(['store_nbr', 'dayofweek'])['log_sales'].mean().to_dict()
base_fam_dow = all_df[train_mask_for_base].groupby(['family', 'dayofweek'])['log_sales'].mean().to_dict()

all_df = pd.merge(all_df, df_store_dow[['date', 'store_nbr', 'store_dow_mean_4w']], on=['date', 'store_nbr'], how='left')
all_df = pd.merge(all_df, df_fam_dow[['date', 'family', 'family_dow_mean_4w']], on=['date', 'family'], how='left')

test_idx = all_df['date'] >= '2017-08-16'
for idx in all_df[test_idx].index:
    st = all_df.at[idx, 'store_nbr']
    fm = all_df.at[idx, 'family']
    dow = all_df.at[idx, 'dayofweek']
    all_df.at[idx, 'store_dow_mean_4w'] = base_store_dow.get((st, dow), np.nan)
    all_df.at[idx, 'family_dow_mean_4w'] = base_fam_dow.get((fm, dow), np.nan)

print('曜日グループ平均特徴量の作成完了！')

## 3. 再帰推論と手元検証 & 提出ファイル作成

完全な実装コードおよび推論スクリプトは `src/dow_group_features.py` に収録されています。
- 手元検証 RMSLE: **`0.39962`**
- Kaggle 本番 Public LB: **`0.38972`**